In [1]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.5 MB/s eta 0:00:00


In [2]:
from docx import Document
from docx.shared import RGBColor
import csv
import random
from pathlib import Path
import spacy
from spacy.tokens import DocBin
from pathlib import Path
import re
import json
from spacy.tokens import Span
from spacy.language import Language

In [3]:
INPUT = "/content/drive/MyDrive/Диплом 2/код/labeling/Complete/Manually labeled data.json"
TRAIN_OUT = "/content/drive/MyDrive/Диплом 2/код/training/Train data.spacy"
TEST_OUT  = "/content/drive/MyDrive/Диплом 2/код/training/Test data.spacy"

In [ ]:
REDACTED_INPUT = "/content/drive/MyDrive/Диплом 2/код/labeling/Complete/Manually labeled data_no_time_date.json"

with open(INPUT, "r", encoding="utf-8") as f:
    data = json.load(f)
for item in data:
    if "entities" in item:
        item["entities"] = [
            ent for ent in item["entities"]
            if ent["label"] not in ["TIME", "DATE"]
        ]
with open(REDACTED_INPUT, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Новий файл збережено: {REDACTED_INPUT}")
INPUT=REDACTED_INPUT

Новий файл збережено: /content/drive/MyDrive/Диплом 2/код/labeling/Complete/Manually labeled data_no_time_date.json


In [ ]:
#Перетворення CSV в .spacy
SPLIT = 0.8
nlp = spacy.blank("en")

def filter_overlaps(spans):
    spans = sorted(spans, key=lambda x: (x.start, -(x.end - x.start)))
    result = []
    occupied = set()
    for span in spans:
        token_range = set(range(span.start, span.end))
        if occupied.intersection(token_range):
            continue
        result.append(span)
        occupied.update(token_range)
    return result

def convert(data, output):
    db = DocBin()
    for item in data:
        text = item["data"]["text"]
        doc = nlp.make_doc(text)
        spans = []
        for ann in item["annotations"]:
            for r in ann["result"]:
                if r["type"] != "labels":
                    continue
                start = r["value"]["start"]
                end = r["value"]["end"]
                label = r["value"]["labels"][0]
                span = doc.char_span(start, end, label=label, alignment_mode="contract")
                if span:
                    spans.append(span)
        spans = filter_overlaps(spans)
        doc.ents = spans
        db.add(doc)
    db.to_disk(output)

data = json.load(open(INPUT, encoding="utf8"))
random.shuffle(data)
split = int(len(data) * SPLIT)
train = data[:split]
test = data[split:]
convert(train, TRAIN_OUT)
convert(test, TEST_OUT)

In [ ]:
!pip install -U spacy
!python -m spacy download en_core_web_sm
!python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency --force

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 83.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:

cfg = Path("config.cfg").read_text()

cfg = cfg.replace("train = null", f"train = {TRAIN_OUT}")
cfg = cfg.replace("dev = null", f"dev = {TEST_OUT}")
Path("config.cfg").write_text(cfg)

2806

In [ ]:
!python -m spacy train config.cfg --output './drive/MyDrive/Диплом 2/код/model' --training.max_steps 10000

ℹ Saving to output directory: drive/MyDrive/Диплом 2/код/model
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     40.87    0.00    0.00    0.00    0.00
  1     200       1458.13   4083.38   57.54   62.08   53.62    0.58
  3     400       3804.21   2142.09   70.32   74.04   66.96    0.70
  5     600        259.37   1181.12   75.30   79.42   71.59    0.75
  8     800        623.18   1145.63   75.34   79.87   71.30    0.75
 10    1000        425.50    849.85   71.89   73.41   70.43    0.72
 14    1200        393.34    653.68   73.77   78.88   69.28    0.74
 18    1400        439.93    506.07   73.08   74.62   71.59    0.73
 23  

In [6]:
DATE_PATTERN = re.compile(
    r"\b("
    r"(June|July|August)\s+1944"
    r"|"
    r"\d{1,2}-\d{1,2}\s+(June|July|August)"
    r"|"
    r"\d{1,2}\s+(June|July|August)"
    r")\b",
    re.IGNORECASE
)

DDAY_PATTERN = re.compile(
    r"\b(D[-\s]?Day|D\s+(plus|minus)\s+\d+)\b",
    re.IGNORECASE
)

TIME_PATTERN = re.compile(r"\b\d{4}B?\b")

RELATIVE_TIME_PATTERN = re.compile(
    r"\b\d+\s+(minutes?|hours?)\s+later\b",
    re.IGNORECASE
)

@Language.component("date_time_regex")
def date_time_regex_component(doc):

    new_ents = list(doc.ents)

    def overlaps(start, end, ents):
        for ent in ents:
            if start < ent.end_char and end > ent.start_char:
                return True
        return False

    # DATE
    for match in DATE_PATTERN.finditer(doc.text):
        start, end = match.start(), match.end()
        if not overlaps(start, end, new_ents):
            span = doc.char_span(start, end, label="DATE")
            if span:
                new_ents.append(span)

    # D-Day
    for match in DDAY_PATTERN.finditer(doc.text):
        start, end = match.start(), match.end()
        if not overlaps(start, end, new_ents):
            span = doc.char_span(start, end, label="D_DAY_REF")
            if span:
                new_ents.append(span)

    # TIME
    for match in TIME_PATTERN.finditer(doc.text):
        start, end = match.start(), match.end()
        if not overlaps(start, end, new_ents):
            span = doc.char_span(start, end, label="TIME")
            if span:
                new_ents.append(span)

    # RELATIVE TIME
    for match in RELATIVE_TIME_PATTERN.finditer(doc.text):
        start, end = match.start(), match.end()
        if not overlaps(start, end, new_ents):
            span = doc.char_span(start, end, label="REL_TIME")
            if span:
                new_ents.append(span)

    doc.ents = new_ents

    return doc
FORBIDDEN_LABELS = {"TIME", "DATE", "REL_TIME", "D_DAY_REF"}

@Language.component("remove_model_time_entities")
def remove_model_time_entities(doc):

    filtered = [ent for ent in doc.ents if ent.label_ not in FORBIDDEN_LABELS]

    doc.ents = filtered
    return doc

In [9]:


# ===================== MODEL =====================

nlp = spacy.load('/content/drive/MyDrive/Диплом 2/код/model/model-last')
nlp.add_pipe("remove_model_time_entities", after="ner")
nlp.add_pipe("date_time_regex", after="remove_model_time_entities")

# ===================== PATHS =====================

root_folder = "/content/drive/MyDrive/Диплом 2/код/AAR_texts_clean"
docx_path = "/content/drive/MyDrive/Диплом 2/код/labeled/"

output_csv = docx_path + "ALL_entities.csv"

# ===================== COLORS =====================

colors = {
    "PER": "00FFFF",
    "ORG": "FFA500",
    "LOC": "0000FF",
    "TIME": "FF00FF",
    "DATE": "DDA0DD",
    "REL_TIME": "8A2BE2",
    "D_DAY_REF": "FF1493",
    "MISC": "808080",
    "LOSSES": "FF6347",
    "ASSETS": "7CFC00",
    "GAINS": "556B2F",
}

# ===================== CSV INIT =====================

with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["file_name", "full_text", "entity_text", "label", "start_char", "end_char"])

    # ===================== WALK THROUGH FILES =====================

    for subdir, _, files in os.walk(root_folder):
        for file in files:

            if not file.endswith(".txt"):
                continue

            file_name = file.replace(".txt", "")
            file_path = os.path.join(subdir, file)

            print(f"Processing: {file_path}")

            # ===================== READ TEXT =====================

            with open(file_path, "r", encoding="utf-8") as f_txt:
                text = f_txt.read()

            doc_spacy = nlp(text)

            # ===================== DOCX  =====================

            doc = Document()

            legend_paragraph = doc.add_paragraph()
            legend_paragraph.add_run("Legend: ")

            for i, (label, hex_color) in enumerate(colors.items()):
                run = legend_paragraph.add_run(label)
                run.font.color.rgb = RGBColor.from_string(hex_color)
                if i < len(colors) - 1:
                    legend_paragraph.add_run(", ")

            legend_paragraph.add_run("\n\n")

            p = doc.add_paragraph()
            cursor = 0

            for ent in doc_spacy.ents:
                if cursor < ent.start_char:
                    p.add_run(text[cursor:ent.start_char])

                run = p.add_run(ent.text)
                hex_color = colors.get(ent.label_, "000000")
                run.font.color.rgb = RGBColor.from_string(hex_color)

                cursor = ent.end_char

            if cursor < len(text):
                p.add_run(text[cursor:])

            doc.save(os.path.join(docx_path, file_name + "_colored.docx"))

            # ===================== WRITE TO GLOBAL CSV =====================
            writer.writerow([file_name, text, "", "", "", ""])

            for ent in doc_spacy.ents:
                writer.writerow([
                    file_name,
                    "",
                    ent.text,
                    ent.label_,
                    ent.start_char,
                    ent.end_char
                ])

print(f"ALL CSV збережено у {output_csv}")

Processing: /content/drive/MyDrive/Диплом 2/код/AAR_texts_clean/6th GB Airborne Division  After Action Reports  June 1944/001_9th Parachute Regiment.txt
Processing: /content/drive/MyDrive/Диплом 2/код/AAR_texts_clean/6th GB Airborne Division  After Action Reports  June 1944/041_9th Parachute Regiment.txt
Processing: /content/drive/MyDrive/Диплом 2/код/AAR_texts_clean/29th US Infantry Division  After Action Reports  June 1944/002_116th Infantry Regiment 2nd Battalion Command Group  Group C.txt
Processing: /content/drive/MyDrive/Диплом 2/код/AAR_texts_clean/29th US Infantry Division  After Action Reports  June 1944/003_115th Infantry Regiment  After Action Report  Colonel Alfred.txt
Processing: /content/drive/MyDrive/Диплом 2/код/AAR_texts_clean/29th US Infantry Division  After Action Reports  June 1944/005_175th Infantry Regiment  After Action Report.txt
Processing: /content/drive/MyDrive/Диплом 2/код/AAR_texts_clean/29th US Infantry Division  After Action Reports  June 1944/008_116th I